<div style="background:linear-gradient(90deg,#C8102E 0%,#7A0019 100%); color:white; padding:28px 32px; border-radius:8px">
  <div style="font-size:0.95em; letter-spacing:2px; opacity:0.85">NOTEBOOK 08 · GENIE SPACE · COMPLETED</div>
  <div style="font-size:2.3em; font-weight:700; margin-top:6px">💬 Self-Serve Cohort Questions with Genie</div>
  <div style="font-size:1.15em; margin-top:10px; max-width:880px; opacity:0.95">
    Hand the finished pre-screening tables to a research coordinator who has never written SQL,
    and let them ask <i>"How many patients are eligible for Trial A?"</i> in plain English.
  </div>
</div>

## 💬 What is a Genie space?

A **Genie space** is a natural-language analytics surface over a set of **governed Unity Catalog
tables** you choose. A user types a question; Genie writes the SQL, runs it on a warehouse, and
shows the answer. The user never sees a `JOIN`, and every query still runs against your
permissioned tables.

This notebook is **setup guidance + curated content**: prepare the tables, then stand up the space
in the UI and seed it with the questions, instructions, and trusted SQL in `genie/`.

In [0]:
%run ./_config

# ⚙️ Configuration: Clinical Trial Pre-Screening (PRE-BUILT)

<div style="background:#f4f6f9; border-left:6px solid #C8102E; padding:14px 18px; border-radius:4px; font-size:0.95em">
This is the <b>companion config notebook</b>. It is <b>pre-built; you do not edit it</b>.
Every other notebook starts with <code>%run ./_config</code> so they all share one
catalog / schema / warehouse and the same read-only OMOP source.<br>
Just set the widgets at the top of <code>00_START_HERE</code> (matching your
bundle's <code>client_catalog</code> / <code>client_schema</code> / <code>warehouse_id</code>
/ <code>source_schema</code>) and re-run.
</div>

Everything here is Unity-Catalog-scoped (no hive_metastore) and reads from widgets, no
hardcoded secrets.

ℹ️ Not creating catalog clinops_catalog (likely pre-provisioned / no CREATE CATALOG): (com.databricks.sql.managedcatalog.acl.UnauthorizedAccessException) PERMISSION_D
✅ Writing to clinops_catalog.clinops_ml
   Reading read-only OMOP source from clinops_catalog.clinops_foundation
   SQL Warehouse: 0123456789abcdef


Helpers ready: fqn(), src(), show_md(), LLM_FAST, LLM_STRONG


## 🏷️ Step 1: Make the tables legible to Genie (COMPLETED)

Genie reads your **table and column comments** as the primary signal for what each field means.
Good comments are the single highest-leverage accuracy lever. `gold_trial_prescreen` is **LONG**
(one row per patient per trial), so we describe that shape: a `trial_id`, an `eligible` flag, and a
`reason`, plus the biomarker fields.

In [0]:
%sql
COMMENT ON TABLE gold_trial_prescreen IS
  'Clinical-trial pre-screening cohort. LONG shape: one row per patient per trial. Each row says whether that patient is eligible for that trial and why, with the final HER2/ER/PR status (structured or NLP-recovered) and a data-provenance flag.';

ALTER TABLE gold_trial_prescreen ALTER COLUMN person_id         COMMENT 'OMOP person identifier, one patient.';
ALTER TABLE gold_trial_prescreen ALTER COLUMN trial_id          COMMENT 'Trial code: A = HER2-positive, B = ER-positive/HER2-negative/postmenopausal, C = triple-negative.';
ALTER TABLE gold_trial_prescreen ALTER COLUMN trial_name        COMMENT 'Human-readable trial name.';
ALTER TABLE gold_trial_prescreen ALTER COLUMN eligible          COMMENT 'TRUE if this patient meets every criterion for this trial.';
ALTER TABLE gold_trial_prescreen ALTER COLUMN reason            COMMENT 'Plain-English explanation of why the patient is or is not eligible for this trial.';
ALTER TABLE gold_trial_prescreen ALTER COLUMN her2_status       COMMENT 'Final HER2 status: Positive, Negative, or Unknown.';
ALTER TABLE gold_trial_prescreen ALTER COLUMN er_status         COMMENT 'Final estrogen-receptor (ER) status: Positive, Negative, or Unknown.';
ALTER TABLE gold_trial_prescreen ALTER COLUMN pr_status         COMMENT 'Final progesterone-receptor (PR) status: Positive, Negative, or Unknown.';
ALTER TABLE gold_trial_prescreen ALTER COLUMN biomarker_source  COMMENT 'Where the biomarker status came from: "structured" = from the measurement table; "nlp" = recovered from a free-text pathology note by a Foundation Model (invisible to plain SQL).';
ALTER TABLE gold_trial_prescreen ALTER COLUMN age_at_dx_years   COMMENT 'Patient age in years at the time of cancer diagnosis.';
ALTER TABLE gold_trial_prescreen ALTER COLUMN menopausal_status COMMENT 'Menopausal status, e.g. Premenopausal or Postmenopausal. Trial B requires Postmenopausal.';
ALTER TABLE gold_trial_prescreen ALTER COLUMN prior_anti_her2   COMMENT 'TRUE if the patient already received anti-HER2 therapy (e.g. trastuzumab). Trial A excludes these.';

## 🛠️ Step 2: Create the Genie space (PRE-BUILT UI steps)

This part happens in the Databricks UI (~2 min):

1. Left sidebar → **Genie** → **+ New**.
2. Name it **`Clinical Trial Pre-Screening`**; describe it
   (*"Ask about patient eligibility for the HER2+ (Trial A), ER+/HER2− (Trial B), and triple-negative (Trial C) trials."*).
3. Pick the **SQL warehouse** from your `warehouse_id` widget (printed below).
4. Under **Tables**, add the gold tables from `{catalog}.{schema}`:
   `gold_trial_prescreen`, `gold_unified_biomarker_profile`, `silver_demographics`.
5. **Save**, the space is live.

<div style="background:#FFF8E1; border-left:6px solid #F2A900; padding:12px 16px; border-radius:4px">
<b>Keep the table count tight.</b> Genie is most accurate with a small, well-described set. Three
is plenty. The exact instructions, sample questions, and trusted SQL to paste into the space are in
<code>genie/genie_space.md</code>.
</div>

In [0]:
show_md(f"""
<div style='background:#f4f6f9; border-left:6px solid #C8102E; padding:14px 18px; border-radius:6px'>
<b>Warehouse ID</b> (Step 3): <code>{WAREHOUSE_ID}</code><br>
<b>Tables to add</b> (Step 4):<br>
&nbsp;&nbsp;• <code>{fqn('gold_trial_prescreen')}</code><br>
&nbsp;&nbsp;• <code>{fqn('gold_unified_biomarker_profile')}</code><br>
&nbsp;&nbsp;• <code>{fqn('silver_demographics')}</code>
</div>
""")

Warehouse ID (Step 3): 0123456789abcdef 
 Tables to add (Step 4): 
  • clinops_catalog.clinops_ml.gold_trial_prescreen 
  • clinops_catalog.clinops_ml.gold_unified_biomarker_profile 
  • clinops_catalog.clinops_ml.silver_demographics

## 🎯 Step 3: Verify the answers before you demo (COMPLETED)

Run the SQL you *expect* Genie to generate so you can confirm its number is right.

In [0]:
%sql
-- The number Genie should return for "How many patients are eligible for Trial A?"
SELECT COUNT(*) AS trial_a_eligible
FROM gold_trial_prescreen
WHERE trial_id = 'A' AND eligible = TRUE;

trial_a_eligible
140


In [0]:
%sql
-- The patients invisible to a structured query. The headline demo question.
SELECT COUNT(*) AS trial_a_eligible_via_nlp
FROM gold_trial_prescreen
WHERE trial_id = 'A' AND eligible = TRUE AND biomarker_source = 'nlp';

trial_a_eligible_via_nlp
31


## 📚 Step 4: Add instructions & trusted SQL to the space (content in genie/)

Two settings turn a decent Genie into an accurate one, both under the space's
**Instructions / Knowledge** panel:
- **General instructions**: plain-language rules (what "eligible" means, that `biomarker_source =
  'nlp'` was recovered from a note, the valid status strings, that the table is one row per patient
  per trial so you filter by `trial_id`).
- **Trusted example SQL**: save 2 to 3 verified queries with plain-English titles; Genie generalizes
  from them. This is the strongest accuracy boost.

**Both are written for you in `genie/genie_space.md`. Copy them into the space.**

## ✅ Takeaway
<div style="background:#E8F5E9; border-left:6px solid #2E7D32; padding:16px 20px; border-radius:6px">
The trial coordinator can now <b>self-serve every cohort question</b>: counts, lists, eligibility,
even "who was found only in the notes", in plain English, on governed, permissioned data, with no
SQL and no engineer in the loop.
</div>

## ▶️ Next step
### → Open the **[coordinator app]($../../app/README)** (`app/`) for the point-and-click pre-screening UI.